<a href="https://colab.research.google.com/github/yeshaa23/zoo-asia-graph-ETSproject/blob/main/notebooks/Data_Joining_ETS_Graf_FIX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()

Saving wikidata.csv to wikidata (1).csv
Saving dbpedia.csv to dbpedia (1).csv


In [2]:
import pandas as pd
import unicodedata
import re

In [5]:
import io

# Load data
df_wd = pd.read_csv(io.StringIO(uploaded['wikidata (1).csv'].decode('utf-8')))
df_db = pd.read_csv(io.StringIO(uploaded['dbpedia (1).csv'].decode('utf-8')))

print("Kolom Wikidata:", df_wd.columns.tolist())
print("Kolom DBpedia :", df_db.columns.tolist())

Kolom Wikidata: ['zoo', 'zooLabel', 'country', 'countryLabel', 'region', 'regionLabel', 'coord', 'inception', 'website']
Kolom DBpedia : ['zooLabel', 'wikidata']


In [6]:
def normalize_text(text):
    if pd.isna(text):
        return None

    text = str(text).strip().lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join([c for c in text if not unicodedata.combining(c)])
    text = re.sub(r"\s+", " ", text)
    return text

In [9]:
if "QID" in df_db.columns:
    df_db = df_db.rename(columns={"QID": "qid"})
df_wd["zooLabel_norm"] = df_wd["zooLabel"].apply(normalize_text)
df_db["zooLabel_norm"] = df_db["zooLabel"].apply(normalize_text)

In [10]:
# Extract qid dari kolom 'zoo'
df_wd["qid"] = df_wd["zoo"].apply(lambda x: str(x).split('/')[-1] if pd.notna(x) and 'http' in str(x) else x)

# Extract qid dari kolom 'wikidata'
df_db["qid"] = df_db["wikidata"].apply(lambda x: str(x).split('/')[-1] if pd.notna(x) and 'http' in str(x) else x)

df_wd["qid"] = df_wd["qid"].astype(str).str.strip()
df_db["qid"] = df_db["qid"].astype(str).str.strip()

In [11]:
# cleaning data
df_wd["qid"] = df_wd["qid"].replace(["nan", "None", ""], pd.NA)
df_db["qid"] = df_db["qid"].replace(["nan", "None", ""], pd.NA)

In [12]:
df_wd = df_wd.drop_duplicates(subset=["qid", "zooLabel_norm"])
df_db = df_db.drop_duplicates(subset=["qid", "zooLabel_norm"])

In [13]:
# gabung data
join_qid = pd.merge(
    df_wd,
    df_db[["qid", "zooLabel", "zooLabel_norm"]],
    on="qid",
    how="left",
    suffixes=("_wd", "_db")
)

print("Hasil join by qid:", join_qid.shape)
print(join_qid.head())

Hasil join by qid: (663, 13)
                                         zoo                      zooLabel_wd  \
0    http://www.wikidata.org/entity/Q2709506                    Singapore Zoo   
1  http://www.wikidata.org/entity/Q114355208        Yokohama fortune aquarium   
2    http://www.wikidata.org/entity/Q2896297                         NegevZoo   
3    http://www.wikidata.org/entity/Q2896297                         NegevZoo   
4  http://www.wikidata.org/entity/Q111464720  The National Aquarium Abu Dhabi   

                               country          countryLabel  \
0  http://www.wikidata.org/entity/Q334             Singapore   
1   http://www.wikidata.org/entity/Q17                 Japan   
2  http://www.wikidata.org/entity/Q801                Israel   
3  http://www.wikidata.org/entity/Q801                Israel   
4  http://www.wikidata.org/entity/Q878  United Arab Emirates   

                                    region   regionLabel  \
0  http://www.wikidata.org/entity/Q2228

In [14]:
unmatched = join_qid[join_qid["zooLabel_db"].isna()].copy()

fallback = pd.merge(
    unmatched.drop(columns=["zooLabel_db", "zooLabel_norm_db"], errors="ignore"),
    df_db[["qid", "zooLabel", "zooLabel_norm"]],
    left_on="zooLabel_norm_wd",
    right_on="zooLabel_norm",
    how="left",
    suffixes=("", "_db2")
)

print("Hasil fallback by label:", fallback.shape)
print(fallback.head())

Hasil fallback by label: (612, 14)
                                         zoo                      zooLabel_wd  \
0    http://www.wikidata.org/entity/Q2709506                    Singapore Zoo   
1  http://www.wikidata.org/entity/Q114355208        Yokohama fortune aquarium   
2  http://www.wikidata.org/entity/Q111464720  The National Aquarium Abu Dhabi   
3    http://www.wikidata.org/entity/Q3156501            Izu Mito Sea Paradise   
4    http://www.wikidata.org/entity/Q1136334        Yangon Zoological Gardens   

                               country          countryLabel  \
0  http://www.wikidata.org/entity/Q334             Singapore   
1   http://www.wikidata.org/entity/Q17                 Japan   
2  http://www.wikidata.org/entity/Q878  United Arab Emirates   
3   http://www.wikidata.org/entity/Q17                 Japan   
4  http://www.wikidata.org/entity/Q836               Myanmar   

                                    region   regionLabel  \
0  http://www.wikidata.org/entity

In [15]:
hasil = join_qid.copy()

mask = hasil["zooLabel_db"].isna()

if "zooLabel_db2" in fallback.columns:
    hasil.loc[mask, "zooLabel_db"] = fallback.loc[mask, "zooLabel_db2"].values


In [16]:
# zooLabel final: prioritas Wikidata
hasil["zooLabel_final"] = hasil["zooLabel_wd"]

kolom_final = [
    "qid",
    "zooLabel_final",
    "countryLabel",
    "inception",
    "regionLabel"
]

df_final = hasil[kolom_final].copy()

# Rename biar rapi
df_final = df_final.rename(columns={
    "zooLabel_final": "zooLabel"
})

In [17]:
# hapus baris tanpa qid atau tanpa nama zoo
df_final = df_final.dropna(subset=["qid", "zooLabel"])

In [18]:
# hapus duplikat final
df_final = df_final.drop_duplicates()

In [19]:
# simpan hasil join final
df_final.to_csv("zoo_asia_final.csv", index=False)

print("\nJumlah data Wikidata :", len(df_wd))
print("Jumlah data DBpedia  :", len(df_db))
print("Jumlah hasil final   :", len(df_final))

print("\nContoh hasil final:")
print(df_final.head(10))


Jumlah data Wikidata : 662
Jumlah data DBpedia  : 300
Jumlah hasil final   : 662

Contoh hasil final:
           qid                         zooLabel                countryLabel  \
0     Q2709506                    Singapore Zoo                   Singapore   
1   Q114355208        Yokohama fortune aquarium                       Japan   
2     Q2896297                         NegevZoo                      Israel   
4   Q111464720  The National Aquarium Abu Dhabi        United Arab Emirates   
5     Q3156501            Izu Mito Sea Paradise                       Japan   
6     Q1136334        Yangon Zoological Gardens                     Myanmar   
7     Q3108672                      Yerevan Zoo                     Armenia   
8     Q2452280                         Baku Zoo                  Azerbaijan   
9     Q1413736             Tama Zoological Park                       Japan   
10    Q3182046                     Shanghai Zoo  People's Republic of China   

               inception   

In [23]:
from google.colab import files
files.download("zoo_asia_final.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>